# Training the GST Regulatory LLM -- Colab

Trains the 131.5M param model on the GST regulatory corpus, pulled from
Hugging Face (`Tharun007/gst-rulings-corpus`) via `huggingface_hub` --
same source and verification as the Kaggle notebook. No Kaggle API
credentials needed here.

**Colab's local disk does NOT survive a session disconnect.** Only
checkpoints are stored on mounted Google Drive, since losing hours of
training progress is the expensive failure mode. The dataset itself is
small (~10.6M tokens) and downloads fresh from HF in seconds each
session -- caching it in Drive isn't worth the quota or the stale-cache
risk, so it lives on local disk instead.


In [42]:
!pip install -q torch numpy tiktoken pandas pyarrow huggingface_hub

In [43]:
# ---- Config -- EDIT THIS ----
MODEL_DIR = "/content/fingpt-131m-project/04_model-architecture/01_main-chapter-code" # Colab path
# MODEL_DIR = "/kaggle/working/fingpt-131m-project/04_model-architecture/01_main-chapter-code" # Kaggle path
# MODEL_DIR = "../../04_model-architecture/01_main-chapter-code" # Local/Jupyter path

CONTEXT_LEN = 1024
BATCH_SIZE = 8
MAX_STEPS = 20000
EVAL_EVERY = 250
EVAL_ITERS = 50
LR = 3e-4
WEIGHT_DECAY = 0.1
PATIENCE = 10  # early-stopping: rounds with no val improvement before stopping

# Memory-saving switches (same approach as the Kaggle notebook) -- needed on
# a 15GB T4 at this batch size / context length / layer count.
USE_AMP = True
USE_ACTIVATION_CHECKPOINTING = True

# ---- HF dataset location -- EDIT IF THE REPO/FILENAMES CHANGE ----
HF_REPO_ID = "Tharun007/gst-rulings-corpus"
HF_TRAIN_FILE = "data/train-00000-of-00001.parquet"
HF_VAL_FILE = "data/validation-00000-of-00001.parquet"
HF_TOKEN = None  # repo is public -- set a token string only if you make it private


## Mount Drive for checkpoint persistence

In [44]:
from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/llm-from-scratch/checkpoints"
DATA_CACHE_DIR = "/content/data"  # local disk -- small corpus, cheap to re-fetch each session

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATA_CACHE_DIR, exist_ok=True)
print(f"Checkpoints -> {CHECKPOINT_DIR}")
print(f"Data cache  -> {DATA_CACHE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints -> /content/drive/MyDrive/llm-from-scratch/checkpoints
Data cache  -> /content/data


## Download corpus from Hugging Face

Uses `hf_hub_download` (not the Kaggle API, and not a raw `requests.get`
on a hand-built URL -- that combination is what produced an HTML-saved-
as-.parquet failure earlier in this project). Each downloaded file's
magic bytes are checked before pandas touches it, so a bad fetch throws a
clear error here instead of a confusing `ArrowInvalid` three cells later.

In [45]:
import pandas as pd
from huggingface_hub import hf_hub_download

def _verify_parquet_magic(path):
    """Parquet files start and end with the 4 bytes b'PAR1'. A bad fetch
    (HTML error page, truncated download, etc.) won't have this -- catch
    it here instead of a confusing ArrowInvalid from pandas later."""
    with open(path, "rb") as f:
        head = f.read(4)
        f.seek(-4, os.SEEK_END)
        tail = f.read(4)
    if head != b"PAR1" or tail != b"PAR1":
        preview = open(path, "rb").read(300)
        raise ValueError(
            f"{path} is not a valid parquet file (header={head!r}, footer={tail!r}). "
            f"First 300 bytes:\n{preview}"
        )

def _download_and_verify(filename):
    local_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
        cache_dir=DATA_CACHE_DIR,
    )
    _verify_parquet_magic(local_path)
    return local_path

train_path = _download_and_verify(HF_TRAIN_FILE)
val_path = _download_and_verify(HF_VAL_FILE)
print(f"Train file -> {train_path}")
print(f"Val file   -> {val_path}")

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)

assert "text" in train_df.columns, f"Expected a 'text' column, got {list(train_df.columns)}"
print(f"Train rows: {len(train_df):,} | Val rows: {len(val_df):,}")
print("Sample:", train_df["text"].iloc[0][:200])


Train file -> /content/data/datasets--Tharun007--gst-rulings-corpus/snapshots/f64cd9c5329d26a8b2d2a342ede616d57ed6c746/data/train-00000-of-00001.parquet
Val file   -> /content/data/datasets--Tharun007--gst-rulings-corpus/snapshots/f64cd9c5329d26a8b2d2a342ede616d57ed6c746/data/validation-00000-of-00001.parquet
Train rows: 2,150 | Val rows: 238
Sample: GUJARAT AUTHORITY FOR ADVANCE RULING
GOODS AND SERVICES TAX
D/5, RAJYA KAR BHAVAN, ASHRAM ROAD,
AHMEDABAD – 380 009.
ADVANCE RULING NO. GUJ/GAAR/R/63/2020
(IN APPLICATION NO. Advance Ruling/SGST&CGST/


## Tokenize + pack

In [46]:
import numpy as np
import tiktoken

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token

def tokenize_and_pack(texts, context_len):
    all_ids = []
    for t in texts:
        all_ids.extend(enc.encode(t))
        all_ids.append(EOT)
    arr = np.array(all_ids, dtype=np.uint16)
    n_seq = len(arr) // context_len
    return arr[: n_seq * context_len].reshape(n_seq, context_len)

train_data = tokenize_and_pack(train_df["text"].tolist(), CONTEXT_LEN)
val_data = tokenize_and_pack(val_df["text"].tolist(), CONTEXT_LEN)
print(f"Train sequences: {train_data.shape[0]:,}")
print(f"Val sequences:   {val_data.shape[0]:,}")


Train sequences: 9,341
Val sequences:   1,033


## Model architecture

In [47]:
import sys
sys.path.insert(0, MODEL_DIR)
from model import GPTConfig, GPTModel


In [48]:
cfg = GPTConfig(context_length=CONTEXT_LEN)
model = GPTModel(cfg)
print(f"Total parameters: {model.num_params():,}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
model = model.to(device)


Total parameters: 131,497,728
Device: cuda


## Training loop

In [49]:
def get_batch(data, batch_size, device):
    idx = np.random.randint(0, data.shape[0], size=batch_size)
    seqs = torch.from_numpy(data[idx].astype(np.int64))
    x = seqs[:, :-1].contiguous()
    y = seqs[:, 1:].contiguous()
    return x.to(device), y.to(device)


def configure_optimizer(model, weight_decay, lr):
    decay_params, no_decay_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        (decay_params if param.dim() >= 2 else no_decay_params).append(param)
    return torch.optim.AdamW([
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=lr, betas=(0.9, 0.95))


@torch.no_grad()
def estimate_loss(model, data, batch_size, device, eval_iters=50):
    model.eval()
    losses = torch.zeros(eval_iters)
    for i in range(eval_iters):
        x, y = get_batch(data, batch_size, device)
        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                _, loss = model(x, y, use_activation_checkpointing=USE_ACTIVATION_CHECKPOINTING)
        else:
            _, loss = model(x, y, use_activation_checkpointing=USE_ACTIVATION_CHECKPOINTING)
        losses[i] = loss.item()
    model.train()
    return losses.mean().item()


def save_checkpoint(path, model, optimizer, step, best_val_loss, no_improve_count, cfg):
    # no_improve_count is saved here -- the original version of this notebook
    # dropped it, which silently reset your early-stopping patience counter
    # to 0 on every resume. Fixed.
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "best_val_loss": best_val_loss,
        "no_improve_count": no_improve_count,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "config": vars(cfg),
    }, path)


In [50]:
# Resume support -- if a checkpoint already exists in CHECKPOINT_DIR (e.g.
# from a previous session that got cut off), pick up from there instead
# of starting over.
# Resume support
import os as _os

last_ckpt_path = _os.path.join(CHECKPOINT_DIR, "last.pt")
optimizer = configure_optimizer(model, WEIGHT_DECAY, LR)

start_step = 0
best_val_loss = float("inf")
no_improve_count = 0

if _os.path.exists(last_ckpt_path):
    print(f"Found existing checkpoint at {last_ckpt_path} -- resuming.")

    # IMPORTANT:
    # This checkpoint is trusted because it was created by our own training script.
    ckpt = torch.load(
        last_ckpt_path,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    start_step = ckpt["step"]
    best_val_loss = ckpt["best_val_loss"]
    no_improve_count = ckpt.get("no_improve_count", 0)

    if "torch_rng_state" in ckpt:
        torch.set_rng_state(
            ckpt["torch_rng_state"].to(torch.uint8).cpu()
        )

    if "numpy_rng_state" in ckpt:
        np.random.set_state(ckpt["numpy_rng_state"])

    print(
        f"  Resumed at step {start_step}, "
        f"best_val_loss so far: {best_val_loss:.4f}, "
        f"no_improve_count: {no_improve_count}"
    )

else:
    print("No existing checkpoint -- starting fresh.")

Found existing checkpoint at /content/drive/MyDrive/llm-from-scratch/checkpoints/last.pt -- resuming.
  Resumed at step 12250, best_val_loss so far: 2.7641, no_improve_count: 10


In [51]:
import time

model.train()
t0 = time.time()
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
grad_norm = torch.tensor(0.0)  # placeholder so the first eval print never hits an undefined name

for step in range(start_step, MAX_STEPS):
    x, y = get_batch(train_data, BATCH_SIZE, device)

    optimizer.zero_grad(set_to_none=True)
    if USE_AMP:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            _, loss = model(x, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        _, loss = model(x, y)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    if step % EVAL_EVERY == 0 or step == MAX_STEPS - 1:
        train_loss_est = estimate_loss(model, train_data, BATCH_SIZE, device, EVAL_ITERS)
        val_loss = estimate_loss(model, val_data, BATCH_SIZE, device, EVAL_ITERS)
        elapsed = time.time() - t0
        gpu_allocated = torch.cuda.memory_allocated() / 1024**3
        gpu_reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"step {step:6d} | train_loss {train_loss_est:.4f} | "
              f"val_loss {val_loss:.4f} | grad {grad_norm:.2f} | "
              f"VRAM {gpu_allocated:.2f}/{gpu_reserved:.2f} GB | {elapsed:.0f}s elapsed")

        save_checkpoint(last_ckpt_path, model, optimizer, step, best_val_loss,
                         no_improve_count, cfg)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve_count = 0
            save_checkpoint(_os.path.join(CHECKPOINT_DIR, "best.pt"),
                             model, optimizer, step, best_val_loss, no_improve_count, cfg)
            print(f"  New best val_loss: {best_val_loss:.4f} -- saved best.pt")
        else:
            no_improve_count += 1
            print(f"  No improvement ({no_improve_count}/{PATIENCE})")

        if no_improve_count >= PATIENCE:
            print(f"\nEarly stopping at step {step} -- best.pt is your model, not last.pt.")
            break

        torch.cuda.empty_cache()

print("\nTraining complete (or interrupted -- re-run this cell to resume from last.pt).")


/tmp/ipykernel_505/1078296452.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


step  12250 | train_loss 1.8753 | val_loss 2.8574 | grad 1.06 | VRAM 3.24/11.92 GB | 20s elapsed
  No improvement (11/10)

Early stopping at step 12250 -- best.pt is your model, not last.pt.

Training complete (or interrupted -- re-run this cell to resume from last.pt).


In [53]:
import torch
import os # Import os module explicitly if not already available in this scope

best_ckpt_path = os.path.join(CHECKPOINT_DIR, "best.pt") # Define best_ckpt_path
last_ckpt_path = os.path.join(CHECKPOINT_DIR, "last.pt") # Define last_ckpt_path for consistency and clarity

best = torch.load(
    best_ckpt_path,
    map_location="cpu",
    weights_only=False
)

last = torch.load(
    last_ckpt_path,
    map_location="cpu",
    weights_only=False
)

print("BEST.PT")
print("step:", best["step"])
print("val loss:", best["best_val_loss"])

print("\nLAST.PT")
print("step:", last["step"])
print("best val loss:", last["best_val_loss"])

BEST.PT
step: 11250
val loss: 2.7640950679779053

LAST.PT
step: 12250
best val loss: 2.7640950679779053
